# Esercitazione

Importa le diverse librerie

In [ ]:
import datasets
import re
from collections import Counter
import pandas as pd
import numpy as np

## Preparazione dati e analisi del testo

Importa i dati da un archivio
dei contenuti di Wikipedia e messo a disposizione sulla piattaforma
[Hugging Face](https://huggingface.co/datasets/Salesforce/wikitext) (Merity et al., 2016).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

Utilizza prima solo i dati del dataset *test*. In un secondo momento puoi ripetere l'esercitazione con il dataset *train*, cambiando il nome tra parentesi quadre.

In [ ]:
data = dataset["test"]
testo = " ".join(data["text"])
print("Lunghezza testo:", len(testo))

### Tokenizzazione

Crea una lista delle parole presenti.

In [ ]:
tokens = re.findall(r"\b\w+\b", testo.lower())
tokens[:10]

## Modello di ordine 0



### Training

Cosa apprende il modello? Nella fase di training calcola le frequenza di ogni **parola tipo** presente nel testo. Il dataframe *df* creato contiene le informazioni apprese da nostro semplice modello di liguaggio naturale.

In [ ]:
tokens = re.findall(r"\b\w+\b", testo.lower())
l_corpus= len(tokens)
frequenze_type = Counter(tokens)

df = (
    pd.DataFrame(frequenze_type.items(), columns=["type", "frequenza"])
    .sort_values(by="frequenza", ascending=False)
    .reset_index(drop=True)
)
df["probabilità"] = df.frequenza/l_corpus
df.head()

### Predict

Dopo la fase di apprendimento puoi passare a fare la previsione. In questo caso il modello ti restituisce semplicemente la parola più probabile.

In [ ]:
def predictM0(df):
  parola_successiva = df.loc[df["probabilità"].idxmax(), "type"]
  return ("La parola prevista è:", parola_successiva)

In [ ]:
predictM0(df)

### Osservazioni

Cosa non considera un modello di ordine 0?

Inserisci qui le tue risposte

## Modello di ordine 1

Considera ora la probabilità di ogni bigramma e non più della singola parola type.

### Training

Costruisci i bigrammi cioè le coppie di parole

In [ ]:
bigrammi = list(zip(tokens[:-1], tokens[1:]))
print(bigrammi[1:10])

Calcola la frequenza di ogni bigramma tipo, quindi dell'evento intersezione.

In [ ]:
frequenze_bigrammi = Counter(bigrammi)
print(frequenze_bigrammi)

Costruisci una tabella con le seguenti colonne: parola successiva $v_2$, parola precedente $v_1$ e la probabilità condizionata di $v_2$ nota la parola precedente $v_1$.

In [ ]:
righe = []

for (w1, w2), count in frequenze_bigrammi.items():
    freq_w1 = frequenze_type[w1]
    prob_w1 = freq_w1/l_corpus
    prob_w1_et_w2 = count / (l_corpus-1)
    righe.append([w1, w2, prob_w1_et_w2, prob_w1, count/freq_w1])

df_bigrammi = pd.DataFrame(
    righe,
    columns=[
        "w1",
        "w2",
        "P(w1_et_w2)",
        "P(w1)",
        "P(w2|w1)"
    ]
)

df_bigrammi.head()

### Predict
Genera la parola successiva in base alla probabilità condizionata.

In [ ]:
def predictM1(parola, df_bigrammi):
    # Se la parola non è nel dizionario, scegliamo una parola a caso basandoci sulle frequenze generali
    if parola not in df_bigrammi.w1.values:
        return np.random.choice(df_bigrammi.w1.values)

    df_possibili = df_bigrammi[df_bigrammi.w1 == parola]
    opzioni = df_possibili.w2.values
    probabilita = df_possibili["P(w2|w1)"].values
    succ = np.random.choice(opzioni, p=probabilita)

    return succ

In [ ]:
print(predictM1("the", df_bigrammi))

### L'autoregressione

Proviamo a generare un testo di n parole.

In [ ]:
def genera_testo_markov1(parola_iniziale, n_parole, corpus):
    parola = parola_iniziale
    risultato = [parola]

    for _ in range(n_parole - 1):
        next_word = predictM1(parola, corpus)
        if next_word is None:
            break
        risultato.append(next_word)
        parola = next_word

    return " ".join(risultato)


In [ ]:
print(genera_testo_markov1("swim", 12, df_bigrammi))

### Osservazioni

Cosa osservi? Il modello ha dimenticato l'inizio della frase?

Inserisci la tua risposta.

# Fonti

Linguistica computazionale, A. Lenci, S. Auriemma, M. Miliani, Hoeppli